Import thư viện và load dữ liệu



In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

def load_file(filename):
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")
    return joblib.load(path)

X_train = load_file("X_train.pkl")
X_test = load_file("X_test.pkl")
y_train = np.asarray(load_file("y_train.pkl")).astype(int)
y_test = np.asarray(load_file("y_test.pkl")).astype(int)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

def show_label_distribution(y, name):
    labels, counts = np.unique(y, return_counts=True)
    print(f"\n{name}")
    for label, count in zip(labels, counts):
        label_name = "Spam" if label == 1 else "Ham"
        print(f"{label} ({label_name}): {count} mẫu - {count / len(y) * 100:.2f}%")

show_label_distribution(y_train, "Train label distribution")
show_label_distribution(y_test, "Test label distribution")

X_train: (4396, 10000)
X_test : (1100, 10000)
y_train: (4396,)
y_test : (1100,)

Train label distribution
0 (Ham): 3303 mẫu - 75.14%
1 (Spam): 1093 mẫu - 24.86%

Test label distribution
0 (Ham): 827 mẫu - 75.18%
1 (Spam): 273 mẫu - 24.82%


Class Multinomial Naive Bayes 

Naive Bayes học bằng xác suất:

- `P(Ham)`, `P(Spam)` gọi là prior probability
- `P(feature | Ham)`, `P(feature | Spam)` gọi là conditional probability
- Dự đoán class có posterior probability cao hơn

`alpha` là tham số smoothing, giúp tránh xác suất bằng 0.


In [3]:
class MultinomialNaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        self.classes_ = np.unique(y)

        n_samples, n_features = X.shape
        self.class_log_prior_ = np.zeros(len(self.classes_))
        self.feature_log_prob_ = np.zeros((len(self.classes_), n_features))

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]

            # Prior probability: log P(class)
            self.class_log_prior_[i] = np.log(X_c.shape[0] / n_samples)

            # Conditional probability: log P(feature | class)
            feature_count = np.asarray(X_c.sum(axis=0)).ravel()
            smoothed_count = feature_count + self.alpha
            self.feature_log_prob_[i] = np.log(smoothed_count / smoothed_count.sum())

        return self

    def predict_log_proba(self, X):
        joint = X.dot(self.feature_log_prob_.T) + self.class_log_prior_
        joint = np.asarray(joint)

        # log-sum-exp để chuẩn hóa xác suất
        max_log = np.max(joint, axis=1, keepdims=True)
        log_sum = max_log + np.log(np.sum(np.exp(joint - max_log), axis=1, keepdims=True))
        return joint - log_sum

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        log_proba = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_proba, axis=1)]

 Hàm đánh giá tự viết

In [4]:
def confusion_values(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))

    return TN, FP, FN, TP

def evaluate(y_true, y_pred):
    TN, FP, FN, TP = confusion_values(y_true, y_pred)

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP
    }

def log_loss_score(y_true, y_proba, classes):
    class_to_index = {c: i for i, c in enumerate(classes)}
    true_index = np.array([class_to_index[y] for y in y_true])

    p = y_proba[np.arange(len(y_true)), true_index]
    p = np.clip(p, 1e-15, 1 - 1e-15)

    return float(-np.mean(np.log(p)))

Thực nghiệm nhiều lần: so sánh alpha

In [11]:
alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]

rows = []

for alpha in alphas:
    model = MultinomialNaiveBayesFromScratch(alpha=alpha)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    metrics = evaluate(y_test, y_pred)
    loss = log_loss_score(y_test, y_proba, model.classes_)

    rows.append({
        "alpha": alpha,
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1-score": metrics["F1-score"],
        "Log Loss": loss,
        "FP": metrics["FP"],
        "FN": metrics["FN"]
    })

alpha_table = pd.DataFrame(rows)
alpha_table

,alpha,Accuracy,Precision,Recall,F1-score,Log Loss,FP,FN
0,0.01,0.989091,0.996198,0.959707,0.977612,0.030185,1,11
1,0.05,0.987273,0.988679,0.959707,0.973978,0.027854,3,11
2,0.10,0.989091,0.988764,0.967033,0.977778,0.028431,3,9
3,0.50,0.988182,0.992424,0.959707,0.975791,0.039815,2,11
4,1.00,0.982727,0.996094,0.934066,0.964083,0.056694,1,18
5,2.00,0.954545,0.995556,0.820513,0.899598,0.092670,1,49


Train mô hình cuối cùng

In [7]:
final_alpha = 0.1

final_model = MultinomialNaiveBayesFromScratch(alpha=final_alpha)
final_model.fit(X_train, y_train)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

y_train_proba = final_model.predict_proba(X_train)
y_test_proba = final_model.predict_proba(X_test)

print("Đã train và predict với alpha =", final_alpha)

Đã train và predict với alpha = 0.1


Đánh giá mô hình cuối cùng

In [8]:
train_metrics = evaluate(y_train, y_train_pred)
test_metrics = evaluate(y_test, y_test_pred)

train_loss = log_loss_score(y_train, y_train_proba, final_model.classes_)
test_loss = log_loss_score(y_test, y_test_proba, final_model.classes_)

result_table = pd.DataFrame({
    "Metric": [
        "Train Accuracy", "Test Accuracy",
        "Train Precision", "Test Precision",
        "Train Recall", "Test Recall",
        "Train F1-score", "Test F1-score",
        "Train Log Loss", "Test Log Loss",
        "TN", "FP", "FN", "TP"
    ],
    "Value": [
        train_metrics["Accuracy"], test_metrics["Accuracy"],
        train_metrics["Precision"], test_metrics["Precision"],
        train_metrics["Recall"], test_metrics["Recall"],
        train_metrics["F1-score"], test_metrics["F1-score"],
        train_loss, test_loss,
        test_metrics["TN"], test_metrics["FP"],
        test_metrics["FN"], test_metrics["TP"]
    ]
})

result_table

,Metric,Value
0,Train Accuracy,0.992948
1,Test Accuracy,0.989091
2,Train Precision,0.991667
3,Test Precision,0.988764
4,Train Recall,0.979872
5,Test Recall,0.967033
6,Train F1-score,0.985734
7,Test F1-score,0.977778
8,Train Log Loss,0.017246
9,Test Log Loss,0.028431


Confusion Matrix

In [9]:
confusion_matrix_table = pd.DataFrame(
    [
        [test_metrics["TN"], test_metrics["FP"]],
        [test_metrics["FN"], test_metrics["TP"]],
    ],
    index=["Actual Ham (0)", "Actual Spam (1)"],
    columns=["Predicted Ham (0)", "Predicted Spam (1)"]
)

confusion_matrix_table

,Predicted Ham (0),Predicted Spam (1)
Actual Ham (0),824,3
Actual Spam (1),9,264


Nhận xét kết quả

Mô hình được xem là tốt nếu:

- Test Accuracy cao
- Precision cao: ít email Ham bị chặn nhầm thành Spam
- Recall cao: bắt được nhiều email Spam thật
- F1-score cao: cân bằng tốt Precision và Recall
- Test Log Loss thấp: xác suất dự đoán tốt
- Train và Test không chênh lệch quá nhiều: ít dấu hiệu overfitting

Vì Naive Bayes không train bằng gradient descent nên không có loss theo epoch như SVM.  
Log Loss ở đây dùng để đánh giá xác suất dự đoán sau khi train.
